## Simple MDP

#### Import packages

In [ ]:
import numpy as np
import mdptoolbox
import mdptoolbox.example
import gymnasium as gym

#### Define MDP

In [12]:
import numpy as np


# ============================================================
# 1. Environment
# ============================================================

class SimpleLinearMDP:
    def __init__(
        env,
        n,
        mu_theta,
        Sigma_theta,
        alpha,
        beta,
        gamma,
        sigma_s,
        sigma_r,
        mu_s0,
        sigma_s0,
        seed=None,
    ):
        env.n = n
        env.rng = np.random.default_rng(seed)

        env.alpha = alpha
        env.beta = beta
        env.gamma = gamma
        env.sigma_s = sigma_s
        env.sigma_r = sigma_r
        env.mu_s0 = mu_s0
        env.sigma_s0 = sigma_s0

        # true person-specific reward parameters
        env.theta = env.rng.multivariate_normal(mu_theta, Sigma_theta, size=n)

        # current state
        env.s = None

    def reset(env):
        env.s = env.rng.normal(env.mu_s0, env.sigma_s0, size=env.n)
        return env.s.copy()

    def step(env, a):
        a = np.asarray(a).astype(float)

        X = np.column_stack([
            np.ones(env.n),
            env.s,
            a,
            env.s * a
        ])

        reward_mean = np.sum(env.theta * X, axis=1)
        reward = reward_mean + env.rng.normal(0, env.sigma_r, size=env.n)

        next_state_mean = env.alpha + env.beta * env.s + env.gamma * a
        next_state = next_state_mean + env.rng.normal(0, env.sigma_s, size=env.n)

        env.s = next_state
        return reward, next_state.copy()


# ============================================================
# 2. Helpers
# ============================================================

def feature_vector(s, a):
    return np.array([1.0, s, float(a), s * float(a)])


def posterior_from_precision(Lambda, b):
    Sigma = np.linalg.inv(Lambda)
    mu = Sigma @ b
    return mu, Sigma


def sample_theta(rng, mu, Sigma):
    return rng.multivariate_normal(mu, Sigma)


# ============================================================
# 3. Unpooled Thompson Sampling
# ============================================================

class UnpooledTS:
    def __init__(alg, n, d=4, lambda_prior=1.0, seed=None):
        alg.n = n
        alg.d = d
        alg.lambda_prior = lambda_prior
        alg.rng = np.random.default_rng(seed)

        alg.Lambda = np.array([lambda_prior * np.eye(d) for _ in range(n)])
        alg.b = np.zeros((n, d))

    def select_actions(alg, s):
        a = np.zeros(alg.n, dtype=int)

        for i in range(alg.n):
            mu_i, Sigma_i = posterior_from_precision(alg.Lambda[i], alg.b[i])
            theta_tilde = sample_theta(alg.rng, mu_i, Sigma_i)

            r0 = feature_vector(s[i], 0) @ theta_tilde
            r1 = feature_vector(s[i], 1) @ theta_tilde
            a[i] = int(r1 > r0)

        return a

    def update(alg, s, a, r):
        for i in range(alg.n):
            x = feature_vector(s[i], a[i])
            alg.Lambda[i] += np.outer(x, x)
            alg.b[i] += x * r[i]


# ============================================================
# 4. Pooled Thompson Sampling
# ============================================================

class PooledTS:
    def __init__(alg, n, d=4, lambda_prior=1.0, seed=None):
        alg.n = n
        alg.d = d
        alg.lambda_prior = lambda_prior
        alg.rng = np.random.default_rng(seed)

        alg.Lambda = lambda_prior * np.eye(d)
        alg.b = np.zeros(d)

    def select_actions(alg, s):
        mu, Sigma = posterior_from_precision(alg.Lambda, alg.b)
        theta_tilde = sample_theta(alg.rng, mu, Sigma)

        a = np.zeros(alg.n, dtype=int)
        for i in range(alg.n):
            r0 = feature_vector(s[i], 0) @ theta_tilde
            r1 = feature_vector(s[i], 1) @ theta_tilde
            a[i] = int(r1 > r0)

        return a

    def update(alg, s, a, r):
        for i in range(alg.n):
            x = feature_vector(s[i], a[i])
            alg.Lambda += np.outer(x, x)
            alg.b += x * r[i]


# ============================================================
# 5. Empirical Bayes Thompson Sampling
# ============================================================

class EBTS:
    def __init__(alg, n, d=4, lambda_prior=1.0, seed=None):
        alg.n = n
        alg.d = d
        alg.lambda_prior = lambda_prior
        alg.rng = np.random.default_rng(seed)

        # local posteriors from each person's own data
        alg.Lambda_local = np.array([lambda_prior * np.eye(d) for _ in range(n)])
        alg.b_local = np.zeros((n, d))

        # EB hyperparameters for theta_i ~ N(mu_pop, diag(tau2_pop))
        alg.mu_pop = np.zeros(d)
        alg.tau2_pop = np.ones(d)

    def estimate_population_hyperparams(alg):
        local_means = np.zeros((alg.n, alg.d))
        local_vars = np.zeros((alg.n, alg.d))

        for i in range(alg.n):
            mu_i, Sigma_i = posterior_from_precision(alg.Lambda_local[i], alg.b_local[i])
            local_means[i] = mu_i
            local_vars[i] = np.diag(Sigma_i)

        # inverse-variance weighted mean
        for j in range(alg.d):
            w = 1.0 / np.maximum(local_vars[:, j], 1e-8)
            alg.mu_pop[j] = np.sum(w * local_means[:, j]) / np.sum(w)

            # method-of-moments style between-person variance estimate
            centered2 = np.mean((local_means[:, j] - alg.mu_pop[j]) ** 2)
            avg_within = np.mean(local_vars[:, j])
            alg.tau2_pop[j] = max(centered2 - avg_within, 1e-6)

    def posterior_for_person(alg, i):
        mu_local, Sigma_local = posterior_from_precision(alg.Lambda_local[i], alg.b_local[i])

        Sigma_pop_inv = np.diag(1.0 / np.maximum(alg.tau2_pop, 1e-8))
        Sigma_local_inv = np.linalg.inv(Sigma_local)

        Sigma_post = np.linalg.inv(Sigma_local_inv + Sigma_pop_inv)
        mu_post = Sigma_post @ (Sigma_local_inv @ mu_local + Sigma_pop_inv @ alg.mu_pop)

        return mu_post, Sigma_post

    def select_actions(alg, s):
        # update EB hyperparameters from current local posteriors
        alg.estimate_population_hyperparams()

        a = np.zeros(alg.n, dtype=int)
        for i in range(alg.n):
            mu_post, Sigma_post = alg.posterior_for_person(i)
            theta_tilde = sample_theta(alg.rng, mu_post, Sigma_post)

            r0 = feature_vector(s[i], 0) @ theta_tilde
            r1 = feature_vector(s[i], 1) @ theta_tilde
            a[i] = int(r1 > r0)

        return a

    def update(alg, s, a, r):
        for i in range(alg.n):
            x = feature_vector(s[i], a[i])
            alg.Lambda_local[i] += np.outer(x, x)
            alg.b_local[i] += x * r[i]


# ============================================================
# 6. Simulation runner
# ============================================================

def run_algorithm(env, alg, T):
    s = env.reset()
    rewards = np.zeros((env.n, T))
    actions = np.zeros((env.n, T), dtype=int)
    states = np.zeros((env.n, T + 1))
    states[:, 0] = s

    for t in range(T):
        a = alg.select_actions(s)
        r, s_next = env.step(a)

        alg.update(s, a, r)

        rewards[:, t] = r
        actions[:, t] = a
        states[:, t + 1] = s_next

        s = s_next

    return {
        "rewards": rewards,
        "actions": actions,
        "states": states,
        "mean_reward": rewards.mean(),
        "cumulative_mean_reward": rewards.mean(axis=0).cumsum(),
    }


In [14]:

n = 50
T = 100

mu_theta = np.array([1.0, 2.0, 3, -0.25])
Sigma_theta = np.diag([0.5, 0.5, 0.25, 0.25])

env_kwargs = dict(
    n=n,
    mu_theta=mu_theta,
    Sigma_theta=Sigma_theta,
    alpha=0.0,
    beta=0.8,
    gamma=0.4,
    sigma_s=0.5,
    sigma_r=1.0,
    mu_s0=0.0,
    sigma_s0=1.0,
)

# use same true environment seed for fair comparison
env_unpooled = SimpleLinearMDP(**env_kwargs, seed=123)
env_pooled = SimpleLinearMDP(**env_kwargs, seed=123)
env_eb = SimpleLinearMDP(**env_kwargs, seed=123)

alg_unpooled = UnpooledTS(n=n, d=4, lambda_prior=1.0, seed=1)
alg_pooled = PooledTS(n=n, d=4, lambda_prior=1.0, seed=1)
alg_eb = EBTS(n=n, d=4, lambda_prior=1.0, seed=1)

res_unpooled = run_algorithm(env_unpooled, alg_unpooled, T)
res_pooled = run_algorithm(env_pooled, alg_pooled, T)
res_eb = run_algorithm(env_eb, alg_eb, T)

print("Unpooled TS mean reward:", res_unpooled["mean_reward"])
print("Pooled TS mean reward:  ", res_pooled["mean_reward"])
print("EB TS mean reward:      ", res_eb["mean_reward"])

Unpooled TS mean reward: 7.502689116433152
Pooled TS mean reward:   7.601961739302057
EB TS mean reward:       7.577259641663431
